# MedGENIE trên Kaggle 2xT4 — Môi trường tương đương Dockerfile (không xung đột)

Notebook này tái hiện `Dockerfile` trên Kaggle (Python 3.12, CUDA 12.8, `torch 2.10.0+cu128` có sẵn, 2x Tesla T4 sm75) **mà không downgrade torch / CUDA của Kaggle**.

## 0. Bảng mapping Dockerfile → Kaggle

| Dockerfile (cũ, CUDA 11.8 / py3.8) | Kaggle T4 (mới, CUDA 12.8 / py3.12) | Lý do |
|---|---|---|
| `torch` (cu118, không ghim) | **Giữ nguyên** `torch==2.10.0+cu128` của Kaggle (Cell 1 khóa bằng `constraints.txt`) | Downgrade torch trên Kaggle nặng ~2GB, dễ vỡ driver |
| `transformers==4.35.0` | `transformers==4.55.4` | 4.35 quá cũ, không hỗ trợ Llama-3 / Zephyr / Phi-3 trên torch 2.10 + py3.12 |
| `tokenizers==0.13.3` | `tokenizers==0.21.4` | Phải khớp `transformers 4.55.x` (0.13 sẽ vỡ ABI) |
| `peft==0.5.0` | `peft==0.17.1` | 0.5 không tương thích `transformers>=4.5x` / `accelerate>=1.x` |
| `datasets==2.14.5` | `datasets==2.21.0` | Đã kiểm chứng không xung đột torch-related (Cell 3) |
| `huggingface_hub==0.18.0` | `huggingface_hub>=0.30,<1.0` | 0.18 không login / download được model gated mới (Llama-2/3) |
| `accelerate/optimum` (không ghim) | `accelerate==1.13.0` + `optimum` mới | `1.13.0` là bản đã verify OK ở Cell 3 |
| `bitsandbytes` (không ghim) | `bitsandbytes>=0.45` | Hỗ trợ torch 2.10 cu128 + T4 8-bit/4-bit |
| `wandb/gdown/nltk/scipy/tqdm` | `wandb/gdown/nltk/evaluate/scipy/tqdm/python-dotenv/packaging/ninja/sentencepiece/protobuf` mới | `scipy==1.10.1` vỡ với `numpy 2.x` → để pip tự resolve |
| `flash-attn --no-build-isolation` | **BỎ QUA trên T4** (giải thích ở Cell 4) | flash-attn 2.x yêu cầu sm80+ (Ampere+). T4 là sm75 → build từ source mất hàng giờ, dễ lỗi. vLLM đã dùng XFormers/Triton, transformers dùng `sdpa`/`eager` là đủ |
| `vllm` (không ghim) | **`vllm==0.10.2` trong venv riêng `/kaggle/temp/vllm-env`** (Cell 4–5) | `pip install vllm` mới nhất (=0.24, CUDA 13 default) gây `ImportError: libcudart.so.13` trên Kaggle CUDA 12.8. 0.10.2 là bản ổn định cuối còn default wheel CUDA 12.8 + hỗ trợ T4 + `transformers>=4.55.2` |

## Thứ tự chạy

1. Cell 1: khóa torch/CUDA của Kaggle → `constraints.txt` (GIỮ NGUYÊN, đã chạy OK).
2. Cell 2: cài stack chính trong kernel Kaggle (đã ghim đúng bản verify OK).
3. Cell 3: verify main env (GIỮ NGUYÊN).
4. Cell 4–5: cài + verify vLLM **cách ly** (THAY Cell 4 cũ bị lỗi `libcudart.so.13`).
5. Cell 6: smoke test MedGENIE + lệnh mẫu cho 2xT4.

> ⚠️ Sau Cell 2, nếu Kaggle báo “restart the kernel”, hãy **Restart & Run tiếp từ Cell 3** (bình thường khi upgrade `transformers/datasets`).

In [ ]:
%%bash
python - <<'EOF'
import importlib.metadata as md
keep = ["torch","torchvision","torchaudio","numpy","triton","nvidia-cublas-cu12","nvidia-cudnn-cu12"]
lines=[]
for p in keep:
    try: lines.append(f"{p}=={md.version(p)}")
    except md.PackageNotFoundError: pass
open("/kaggle/working/constraints.txt","w").write("\n".join(lines)+"\n")
print("\n".join(lines))
EOF

In [ ]:
# Cell 2 — Stack chính (main kernel): tương đương Dockerfile nhưng modernize cho torch 2.10 + py3.12
# -c constraints.txt đảm bảo KHÔNG đụng tới torch/numpy/triton/cuDNN của Kaggle
# Các bản ghim dưới đây chính là combo đã verify OK ở Cell 3 (transformers 4.55.4 | tokenizers 0.21.4 | peft 0.17.1 | accelerate 1.13.0)
%pip install -q -c /kaggle/working/constraints.txt \
  "transformers==4.55.4" \
  "tokenizers==0.21.4" \
  "huggingface_hub>=0.30,<1.0" \
  "accelerate==1.13.0" \
  "peft==0.17.1" \
  "datasets==2.21.0" \
  "bitsandbytes>=0.45" \
  "optimum" "sentencepiece" "protobuf" "wandb" "gdown" "nltk" "evaluate" "scipy" "tqdm" "python-dotenv" "packaging" "ninja"

In [ ]:
%%bash
pip check | grep -Ei "torch|transformers|tokenizers|datasets|peft|accelerate" || echo "OK - khong co xung dot lien quan"
python -c "
import torch, transformers, tokenizers, datasets, peft, accelerate
print('torch', torch.__version__, torch.version.cuda, torch.cuda.device_count())
print('transformers', transformers.__version__, '| tokenizers', tokenizers.__version__)
print('datasets', datasets.__version__, '| peft', peft.__version__, '| accelerate', accelerate.__version__)
print('capability', torch.cuda.get_device_capability(0))
print('bf16 supported:', torch.cuda.is_bf16_supported())
"

## 1. Vì sao vLLM phải cài riêng + ghim `0.10.2`? Vì sao bỏ `flash-attn`?

### Lỗi Cell 4 cũ: `libcudart.so.13`

- Cell cũ chạy `uv pip install vllm --torch-backend=cu128` → uv resolve ra **`vllm==0.24.0` + `torch==2.11.0+cu128` + `transformers==5.17.0`**.
- Từ v0.20+, wheel default của vLLM build bằng **CUDA 13** (`VLLM_MAIN_CUDA_VERSION=13.0`), trong khi Kaggle T4 chỉ có runtime **CUDA 12.8** → `import vllm` vỡ ngay tại `vllm/_C_stable_libtorch` với `libcudart.so.13: cannot open shared object file`.
- `--torch-backend=cu128` **chỉ chọn bản torch**, không đổi được bản CUDA của chính wheel vLLM → phải ghim bản còn default CUDA 12.8 **hoặc** dùng wheel `+cu128` tường minh.
- Thêm nữa `transformers 5.17.0` trong venv sẽ lệch với `4.55.4` ở main env, và venv `--seed` thiếu `wrapt` gây `ModuleNotFoundError: No module named 'wrapt'`.

### Chốt phương án (đã đối chiếu docs vLLM v0.10.1/0.10.2 + PyPI)

- **`vllm==0.10.2`**: default wheel **CUDA 12.8**, yêu cầu `torch==2.8.0 / torchvision==0.23.0 / transformers>=4.55.2` → khớp hoàn hảo với `transformers==4.55.4` ở main env, hỗ trợ T4 (`compute capability >= 7.0`), Python 3.12.
- **Cài trong venv `/kaggle/temp/vllm-env`**: giữ `torch 2.10` của Kaggle nguyên vẹn; venv dùng `torch 2.8.0+cu128` riêng.
- **T4 tuning bắt buộc**: `VLLM_ATTENTION_BACKEND=XFORMERS` + `VLLM_USE_V1=0` (V0 + XFormers ổn định nhất trên Turing; tránh FlashInfer JIT `invalid argument`), `dtype=half/float16` (T4 sm75 **không có BF16 native**), `quantization=awq` (cấm `awq_marlin` — Marlin lỗi Zero-Point trên sm75), `max_model_len=2048–4096` + `gpu_memory_utilization~0.90` (tránh pre-allocate hết 16GB cho KV cache), `tensor_parallel_size=2` để dùng cả 2xT4.
- **`flash-attn`**: Dockerfile gốc `pip install flash-attn --no-build-isolation` sẽ **build từ source hàng giờ trên Kaggle và vẫn không chạy trên T4** (flash-attn 2.x chỉ hỗ trợ sm80+; T4 cần repo riêng `flash-attention-turing`). MedGENIE không cần nó: vLLM đã có XFormers/Triton, HF transformers dùng `attn_implementation='sdpa'`/`'eager'`. Nếu bắt buộc, dùng wheel T4 build sẵn (`flash_attn-2.8.3-cp312-...-torch2.10-cu128`, `TORCH_CUDA_ARCH_LIST=7.5`) — nhưng **khuyến nghị bỏ qua**.

In [ ]:
%%bash
set -ex
# 0) uv installer (warning '--system has no effect' la vo hai: uv dang bo qua kernel venv de dung /usr/bin/python3.12 — dung nhu mong muon)
pip install -q uv
rm -rf /kaggle/temp/vllm-env
uv venv /kaggle/temp/vllm-env --python 3.12 --seed

# FIX loi 'Error in sitecustomize: No module named wrapt':
# Kaggle dat sitecustomize.py o system (import wrapt), venv moi chua co wrapt nen MOI LAN goi /venv/bin/python deu in loi nay.
# Loi nay KHONG fatal (python van chay tiep, exit 0) nhung gay hoang mang + che log.
# => Cai wrapt TRUOC TIEN bang 'uv pip' (uv khong invoke venv python nen khong dinh loi), sau do moi dung /venv/bin/python.
echo "--- [1/4] pre-install wrapt+setuptools (fix sitecustomize) ---"
uv pip install --python /kaggle/temp/vllm-env/bin/python \
  "wrapt" "setuptools>=77,<80" \
  --index-url https://pypi.org/simple
/kaggle/temp/vllm-env/bin/python -c "import wrapt; print('wrapt OK')"

# 1) Ghim torch cu128 KHOP driver Kaggle (CUDA 12.8) + dung yeu cau cua vLLM 0.10.2 (torch==2.8.0).
#    Khong dung torch 2.10 cua main env trong venv nay. Buoc nay ~800MB, 5-10 phut — KHONG interrupt khi thay im lang.
echo "--- [2/4] installing torch 2.8.0 cu128 ---"
uv pip install --python /kaggle/temp/vllm-env/bin/python \
  "torch==2.8.0" "torchvision==0.23.0" "torchaudio==2.8.0" \
  --index-url https://download.pytorch.org/whl/cu128

# 2) Ghim vLLM 0.10.2 = ban on dinh cuoi con default CUDA 12.8 (tranh loi libcudart.so.13 cua >=0.20).
#    wheels.vllm.ai/0.10.2 la index chinh thuc cho dong 0.10.2 (xem release notes). Buoc nang nhat, 10+ phut.
echo "--- [3/4] installing vllm==0.10.2 ---"
uv pip install --python /kaggle/temp/vllm-env/bin/python \
  "vllm==0.10.2" \
  --extra-index-url https://wheels.vllm.ai/0.10.2/ \
  --extra-index-url https://download.pytorch.org/whl/cu128 \
  --index-strategy unsafe-best-match

# 3) Deps cho benchmark.py (dung uv pip de tranh goi python truc tiep nhieu lan)
echo "--- [4/4] installing benchmark deps ---"
uv pip install --python /kaggle/temp/vllm-env/bin/python \
  "python-dotenv" "huggingface_hub>=0.30" \
  --index-url https://pypi.org/simple

# 4) Smoke import (chua can GPU)
/kaggle/temp/vllm-env/bin/python -c "import vllm, torch; print('vLLM', vllm.__version__, '| torch', torch.__version__, '| cuda', torch.version.cuda)"

In [ ]:
%%bash
# T4-stable flags: XFormers + V0 engine. Export ở mọi cell/shell dùng vLLM.
export VLLM_ATTENTION_BACKEND=XFORMERS
export VLLM_USE_V1=0
echo "== nvidia-smi (phai thay 2x T4, ~15360 MiB, CC 7.5) =="
nvidia-smi --query-gpu=index,name,memory.total,compute_cap --format=csv || nvidia-smi
echo ""
echo "== vLLM + CUDA trong venv =="
/kaggle/temp/vllm-env/bin/python - <<'EOF'
import torch, vllm
print("vLLM:", vllm.__version__)
print("torch:", torch.__version__, "| cuda build:", torch.version.cuda, "| GPUs:", torch.cuda.device_count())
assert torch.cuda.is_available(), "CUDA not available trong venv!"
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i))
print("bf16 supported (torch):", torch.cuda.is_bf16_supported())
print("-> T4 sm75 KHONG co BF16 native: benchmark MedGENIE phai dung dtype='half'/'float16' (da set san trong code mau Cell 6).")
EOF
echo ""
echo "== Kiem tra link CUDA (phai la .so.12, KHONG duoc .so.13) =="
ldd /kaggle/temp/vllm-env/lib/python3.12/site-packages/vllm/_C*.so 2>&1 | grep -Ei "cudart" || echo "(khong thay cudart trong ldd)"
echo "(Ghi chu: ldd bao libtorch/c10 'not found' la BINH THUONG khi goi ldd truc tiep — Python tu nap qua torch.libs luc import. Mien import vllm o tren OK + cudart la .so.12 la dat.)"

In [ ]:
# Cell 6a — Smoke test main env: đúng các import mà icl_reader/benchmark.py + fid_reader dùng
import torch
import transformers, tokenizers, datasets, peft, accelerate
import bitsandbytes, optimum
from transformers import AutoTokenizer
from huggingface_hub import login
from dotenv import load_dotenv
print("main env OK:", transformers.__version__, tokenizers.__version__, datasets.__version__, peft.__version__, accelerate.__version__)
print("torch:", torch.__version__, torch.version.cuda, "| device_count:", torch.cuda.device_count())
# Đừng `from vllm import ...` ở main env — vLLM nằm trong venv riêng. Kiểm tra bằng cell bash bên dưới.

In [ ]:
%%bash
export VLLM_ATTENTION_BACKEND=XFORMERS
export VLLM_USE_V1=0
# Cell 6b — Smoke test vLLM API trong venv
# LUU Y: vLLM >=0.10 da XOA 'use_beam_search' khoi SamplingParams (beam search chuyen sang LLM.beam_search + BeamSearchParams).
# MedGENIE luon dung use_beam_search=False (greedy/sampling, khong beam) nen chi can BO kwarg nay, khong anh huong ket qua.
/kaggle/temp/vllm-env/bin/python - <<'EOF'
from vllm import LLM, SamplingParams
print("vLLM LLM API OK")
# Pattern ICL (benchmark.py, greedy): truoc day co use_beam_search=False -> nay bo
sp = SamplingParams(n=1, temperature=0.0, top_p=1.0, max_tokens=50)
print("SamplingParams ICL OK:", sp.temperature, sp.max_tokens)
# Pattern context-gen (generate_contexts.py, sampling): giu best_of (0.10.2 van ho tro), bo use_beam_search
sp2 = SamplingParams(n=2, temperature=0.9, frequency_penalty=1.95, top_p=1.0, max_tokens=512)
print("SamplingParams gen OK:", sp2.temperature, sp2.max_tokens)
# AWQ path: LLM(model=..., quantization='awq', dtype='half', max_model_len=2048, tensor_parallel_size=2)
# ICL path: LLM(model=..., dtype='half' neu awq else 'auto', max_model_len=4096, tensor_parallel_size=2, enforce_eager=True neu phi)
print("MedGENIE llm_args pattern OK (awq/half/tp=2 cho 2xT4)")
EOF

In [ ]:
%%bash
# Cell 6c — Patch code MedGENIE cho vLLM>=0.10 (BAT BUOC neu repo tren Kaggle van con use_beam_search).
# benchmark.py va generate_contexts.py cu truyen use_beam_search -> se loi y het Cell 6b cu.
# MedGENIE luon dung False nen patch = xoa dong do (idempotent: chay lai nhieu lan van an toan).
# Giu nguyen field CLI --use_beam_search (input_args.py) de lenh cu khong vo.
set -e
echo "--- /kaggle/working co gi? (kiem tra repo da co tren Kaggle chua) ---"
ls /kaggle/working 2>/dev/null || echo "(khong co /kaggle/working — ban dang chay local?)"
BENCH=""; GEN=""
for ROOT in "." "/kaggle/working" "/kaggle/working/medgenie" "/kaggle/input" "$HOME"; do
  [ -d "$ROOT" ] || continue
  [ -z "$BENCH" ] && BENCH=$(find "$ROOT" -maxdepth 4 -name benchmark.py -path "*icl_reader*" 2>/dev/null | head -1)
  [ -z "$GEN" ] && GEN=$(find "$ROOT" -maxdepth 4 -name generate_contexts.py -path "*context_generation*" 2>/dev/null | head -1)
done
echo "benchmark.py: ${BENCH:-KHONG THAY}"
echo "generate_contexts.py: ${GEN:-KHONG THAY}"
if [ -z "$BENCH" ] || [ -z "$GEN" ]; then
  echo "=> Repo MedGENIE chua co tren Kaggle. Clone (hoac upload) truoc roi chay lai cell nay:"
  echo "   git clone https://github.com/disi-unibo-nlp/medgenie.git /kaggle/working/medgenie"
  exit 1
fi
export BENCH GEN
cp "$BENCH" "${BENCH}.bak"; cp "$GEN" "${GEN}.bak"; echo "(da backup *.bak)"
BENCH="$BENCH" GEN="$GEN" /kaggle/temp/vllm-env/bin/python - <<'EOF'
import os, re
for p in [os.environ["BENCH"], os.environ["GEN"]]:
    s = open(p).read()
    s2, n = re.subn(r"[^\n]*use_beam_search\s*=\s*[^,\n]*,?\n", "\n", s)
    if n:
        open(p, "w").write(s2)
    print(os.path.basename(p), f"-> da xoa {n} dong use_beam_search" + (" (repo nay da patch tu truoc)" if n == 0 else ""))
EOF
echo "--- verify (dong NOTE comment thi khong sao, chi so code truyen kwarg) ---"
grep -n "use_beam_search" "$BENCH" "$GEN" | grep -v "#" && echo "VAN CON code use_beam_search -> bao lai" || echo "OK: het use_beam_search trong code SamplingParams"
/kaggle/temp/vllm-env/bin/python -c "import ast, os; [ast.parse(open(p).read()) for p in [os.environ['BENCH'], os.environ['GEN']]]; print('syntax OK')"
echo "Chay lai Cell 6b de confirm, roi moi chay benchmark that."


In [ ]:
%%bash
# Cell 6d — Deps runtime cho benchmark.py / generate_contexts.py TRONG venv vLLM.
# benchmark.py import: torch, dotenv, tqdm, datasets, vllm, transformers, huggingface_hub (+ prompts local).
# generate_contexts.py import: torch, pandas, tqdm, vllm, transformers (HfArgumentParser), datasets.
# Venv hien tai chi co torch+vllm -> thieu datasets/tqdm/pandas -> ModuleNotFoundError: No module named 'datasets'.
# Ghim transformers/tokenizers/datasets BANG main env de tokenizer 2 ben khong lech nhau.
set -e
uv pip install --python /kaggle/temp/vllm-env/bin/python \
  "transformers==4.55.4" "tokenizers==0.21.4" "datasets==2.21.0" \
  "tqdm" "pandas" "huggingface_hub>=0.30" "python-dotenv" \
  --extra-index-url https://download.pytorch.org/whl/cu128 \
  --index-strategy unsafe-best-match
/kaggle/temp/vllm-env/bin/python - <<'EOF'
import torch, vllm, transformers, tokenizers, datasets, tqdm, pandas, dotenv, huggingface_hub
print("venv bench deps OK:", transformers.__version__, tokenizers.__version__, datasets.__version__)
# Mo phong dung thu tu import cua benchmark.py (tru prompts local can cd dung thu muc)
from datasets import load_dataset
from transformers import AutoTokenizer, HfArgumentParser
from vllm import LLM, SamplingParams
print("benchmark imports OK")
EOF

In [ ]:
# Cell 6e — HF authBang Kaggle Secrets (chay 1 lan truoc benchmark; BAT BUOC cho model gated nhu Zephyr/Llama).
# benchmark.py doc HF_KEY (fallback HF_TOKEN). Khong co token -> login cu hoi interactive -> termios.error nhu screenshot.
# Cach tao: Kaggle notebook -> Add-ons -> Secrets -> Add secret ten HF_KEY, dan token HF (can quyen Read + Accept license model), bat Attach.
import os
tok = None
try:
    from kaggle_secrets import UserSecretsClient
    for _name in ("HF_KEY", "HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            tok = UserSecretsClient().get_secret(_name)
        except Exception:
            tok = None
        if tok:
            os.environ["HF_KEY"] = tok
            os.environ["HF_TOKEN"] = tok
            print(f"HF auth OK tu Secrets '{_name}' — HF_KEY/HF_TOKEN da co trong moi truong, cac cell %%bash ke tiep duoc thua huong.")
            break
    if not tok:
        print("Chua thay secret HF. Add-ons -> Secrets -> Add secret (HF_KEY) + Attach, roi chay lai cell.")
except ImportError:
    print("Khong co kaggle_secrets (dang chay local?) — export HF_KEY/HF_TOKEN thu cong truoc khi chay benchmark.")
if os.environ.get("HF_KEY"):
    try:
        from huggingface_hub import whoami
        print("whoami:", whoami().get("name"))
    except Exception as e:
        print("whoami loi:", type(e).__name__, "— kiem tra token / Accept license model gated.")
else:
    print("(chua login — model public thi van chay duoc, model gated se bao 401/GatedRepoError)")


In [ ]:
# Cell 7 — Chuan bi data medqa cho generate_contexts.py (chay 1 lan, kernel chinh).
# generate_contexts doc medqa bang open(data_path) local, KHONG co fallback HF -> thieu --data_path_test se crash.
# Xuat split test tu HF disi-unibo-nlp/medqa-MedGENIE ra JSONL. File nay dung ngay cho pass --no_options (chi can question).
# Pass co-options can raw MedQA co field 'options' rieng (Drive o README): gdown 'https://drive.google.com/uc?id=1ImYUSLk9JbgHXOemfvyiDiirluZHPeQw'
from datasets import load_dataset
ds = load_dataset("disi-unibo-nlp/medqa-MedGENIE", split="test")
ds.to_json("/kaggle/working/medqa_test.jsonl")
print(len(ds), "samples -> /kaggle/working/medqa_test.jsonl")
print("cols:", ds.column_names)
